<a href="https://colab.research.google.com/github/paulaprado1904/AlgoritmoEvolutivo/blob/main/AE_delta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import math
import time
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Contador global de avaliações completas da função objetivo.
# É usado como medida de custo computacional e como critério de parada.
GLOBAL_AVALIACOES = 0


# ======================================================
# === Estrutura do indivíduo (representação absoluta) ===
# ======================================================
class Individuo:
    """
    Representa uma solução candidata para o problema de dobramento proteico
    no modelo HP em malha 3D.

    Cada indivíduo é codificado por uma sequência de movimentos absolutos
    em uma malha cúbica tridimensional. A partir dessa sequência, são
    reconstruídas as coordenadas da cadeia e calculadas as métricas de
    avaliação utilizadas pelo algoritmo.
    """
    def __init__(self, movimentos):
        # Cromossomo do indivíduo: sequência de movimentos absolutos
        self.movimentos = movimentos[:]

        # Coordenadas reconstruídas da cadeia
        self.coords = None

        # Número de colisões
        self.nC = 0

        # Número de contatos H-H não consecutivos
        self.nH = 0

        # Valor da função objetivo principal
        self.f = None

        # Métricas de compactação
        self.dmax = None
        self.davg = None

        # Métrica delta usada como critério principal nesta variante
        self.delta = None

        # Energia equivalente da solução
        self.energia = None

    def __str__(self):
        """
        Retorna uma representação textual resumida do indivíduo,
        útil para depuração e inspeção manual.
        """
        f_val = self.f if self.f is not None else 0
        d_max = self.dmax if self.dmax is not None else 0
        d_avg = self.davg if self.davg is not None else 0
        delta_val = self.delta if self.delta is not None else 0
        energia_val = self.energia if self.energia is not None else 0

        return (
            f"f(c): {f_val:3} | "
            f"nH: {self.nH:2} | "
            f"nC: {self.nC:2} | "
            f"dmax: {d_max:5.2f} | "
            f"davg: {d_avg:5.2f} | "
            f"delta: {delta_val:6.2f} | "
            f"Energia: {energia_val} | "
            f"movimentos: {self.movimentos}"
        )


# ======================================================
# === Movimentos absolutos em 3D ===
# ======================================================
def delta_from_move_3d(mov):
    """
    Converte o código de um movimento em seu deslocamento cartesiano
    correspondente na malha 3D.

    Convenção adotada:
    0 -> +x
    1 -> +y
    2 -> +z
    3 -> -z
    4 -> -y
    5 -> -x
    """
    if mov == 0:   # U: +x
        return (1, 0, 0)
    if mov == 1:   # L: +y
        return (0, 1, 0)
    if mov == 2:   # F: +z
        return (0, 0, 1)
    if mov == 3:   # B: -z
        return (0, 0, -1)
    if mov == 4:   # R: -y
        return (0, -1, 0)
    if mov == 5:   # D: -x
        return (-1, 0, 0)
    raise ValueError("mov inválido (esperado 0..5)")


# ======================================================
# === Construção da conformação e contagem de colisões ===
# ======================================================
def construir_coordenadas(ind):
    """
    Reconstrói as coordenadas da cadeia a partir da sequência de movimentos
    do indivíduo.

    Retorna:
    - coords: lista com as posições ocupadas pela cadeia
    - nC: número de colisões

    Neste código, o número de colisões é calculado por pares de sobreposição:
    para uma posição ocupada v vezes, soma-se C(v, 2) = v * (v - 1) / 2.
    """
    x = y = z = 0
    coords = [(0, 0, 0)]
    ocupacao = {(0, 0, 0): 1}

    for mov in ind.movimentos:
        dx, dy, dz = delta_from_move_3d(mov)
        x += dx
        y += dy
        z += dz
        pt = (x, y, z)
        coords.append(pt)
        ocupacao[pt] = ocupacao.get(pt, 0) + 1

    # Número de colisões por pares de sobreposição
    nC = sum(v * (v - 1) // 2 for v in ocupacao.values() if v > 1)

    ind.coords = coords
    ind.nC = nC
    return coords, nC


def avaliar_base(ind, cadeia, rho=1.0):
    """
    Avalia completamente um indivíduo.

    Métricas calculadas:
    - nC: número de colisões
    - nH: número de contatos H-H não consecutivos
    - f(c): função objetivo base
    - dmax: maior distância entre pares H-H válidos
    - davg: média das distâncias entre pares H-H válidos
    - energia: energia equivalente da solução

    Regras:
    - Se houver colisão, a solução é considerada inviável.
    - Ainda assim, dmax e davg são mantidos com penalização, para que a
      informação estrutural não seja totalmente descartada.
    - Se não houver colisão, f(c) = nH.
    """
    global GLOBAL_AVALIACOES
    GLOBAL_AVALIACOES += 1

    coords, nC = construir_coordenadas(ind)
    n = len(cadeia)

    # Salva as coordenadas no indivíduo para uso posterior
    ind.coords = coords

    # --------------------------------------------------
    # Distâncias entre resíduos H não consecutivos
    # --------------------------------------------------
    H_idx = [i for i, a in enumerate(cadeia) if a == "H"]
    dists = []
    m = len(H_idx)

    for a in range(m):
        i = H_idx[a]
        for b in range(a + 1, m):
            j = H_idx[b]

            p1 = coords[i]
            p2 = coords[j]

            # Ignora pares colapsados na mesma posição devido a colisão
            if p1 == p2:
                continue

            # Ignora pares consecutivos na cadeia
            if abs(i - j) == 1:
                continue

            d = math.sqrt(
                (p1[0] - p2[0]) ** 2 +
                (p1[1] - p2[1]) ** 2 +
                (p1[2] - p2[2]) ** 2
            )
            dists.append(d)

    if not dists:
        dmax = float("inf")
        davg = float("inf")
    else:
        dmax = max(dists)
        davg = sum(dists) / len(dists)

    # --------------------------------------------------
    # Caso inviável: há colisões
    # --------------------------------------------------
    if nC > 0:
        ind.nC = nC
        ind.nH = 0
        ind.f = -rho * nC

        # Compactação penalizada por colisão
        ind.dmax = 1 / (dmax + (rho * nC))
        ind.davg = 1 / (davg + (rho * nC))

        ind.delta = None
        ind.energia = -ind.f
        return

    # --------------------------------------------------
    # Caso viável: sem colisões
    # Conta contatos H-H não consecutivos e adjacentes no espaço
    # --------------------------------------------------
    nH = 0
    for i in range(n):
        if cadeia[i] != "H":
            continue
        for j in range(i + 2, n):
            if cadeia[j] != "H":
                continue

            pi = coords[i]
            pj = coords[j]

            # Distância Manhattan entre os resíduos i e j
            manh = (
                abs(pi[0] - pj[0]) +
                abs(pi[1] - pj[1]) +
                abs(pi[2] - pj[2])
            )

            if manh == 1:
                nH += 1

    ind.nC = 0
    ind.nH = nH
    ind.f = nH
    ind.dmax = 1 / dmax
    ind.davg = 1 / davg
    ind.delta = None
    ind.energia = -ind.f


# ======================================================
# === Geração de indivíduo aleatório ===
# ======================================================
def gerar_individuo_aleatorio(n_residuos):
    """
    Gera um indivíduo aleatório com n_residuos - 1 movimentos.

    Nesta implementação, todos os 6 movimentos possíveis podem ser usados
    na geração inicial.
    """
    L = n_residuos - 1
    movimentos = [random.randint(0, 5) for _ in range(L)]
    return Individuo(movimentos)


# ======================================================
# === Recombinação por múltiplos pontos ===
# ======================================================
def crossover_multipontos_2filhos(pai, mae, n_residuos):
    """
    Gera dois filhos por crossover de múltiplos pontos.

    O número de cortes é proporcional ao tamanho da proteína.
    """
    L = len(pai.movimentos)
    if L != len(mae.movimentos):
        raise ValueError("Pais com comprimentos diferentes de cromossomo")

    num_cortes = int(round(n_residuos / 10.0))
    num_cortes = max(1, min(num_cortes, L - 1))
    cortes = sorted(random.sample(range(1, L), num_cortes))

    def build_child(start_with_pai: bool):
        filho_mov = []
        pais = [pai.movimentos, mae.movimentos]
        idx = 0 if start_with_pai else 1
        ini = 0
        for c in cortes + [L]:
            filho_mov.extend(pais[idx][ini:c])
            idx = 1 - idx
            ini = c
        return Individuo(filho_mov)

    return build_child(True), build_child(False)


# ======================================================
# === Mutação gene a gene ===
# ======================================================
def mutacao_por_gene(ind, taxa_mut):
    """
    Aplica mutação gene a gene.

    Cada posição do cromossomo é alterada com probabilidade taxa_mut,
    sendo substituída por um novo movimento aleatório.
    """
    novo = Individuo(ind.movimentos[:])
    for j in range(len(novo.movimentos)):
        if random.random() <= taxa_mut:
            novo.movimentos[j] = random.randint(0, 5)
    return novo


# ======================================================
# === Seleção e sobrevivência baseadas em delta ===
# ======================================================
def selecao_torneio_delta(pop, k=3):
    """
    Seleção por torneio usando delta como critério principal.

    Indivíduos com delta não calculado recebem valor muito baixo,
    reduzindo sua chance de serem escolhidos.
    """
    cand = random.sample(pop, k=min(k, len(pop)))

    def key(ind):
        delta_val = ind.delta if ind.delta is not None else -float("inf")
        return delta_val

    return max(cand, key=key)


def selecao_sobreviventes_mu_plus_lambda_delta(pop, filhos, pop_size, elitismo=1):
    """
    Seleção de sobreviventes no esquema (mu + lambda).

    Estratégia:
    - mantém uma elite da população anterior;
    - combina pais e filhos;
    - remove duplicatas por cromossomo;
    - ordena os indivíduos usando um score composto.

    Observação:
    nesta implementação, o score usa primeiro f e depois delta.
    """
    def score(ind):
        delta = ind.delta if ind.delta is not None else -float("inf")
        f = ind.f if ind.f is not None else -float("inf")
        return (float(f), float(delta))

    elite = sorted(pop, key=score, reverse=True)[:max(0, elitismo)]
    candidatos = pop + filhos

    vistos = set()
    unicos = []

    for ind in sorted(candidatos, key=score, reverse=True):
        key = tuple(ind.movimentos)
        if key in vistos:
            continue
        vistos.add(key)
        unicos.append(ind)

    nova = []
    elite_keys = {tuple(e.movimentos) for e in elite}
    nova.extend(elite)

    for ind in unicos:
        if len(nova) >= pop_size:
            break
        if tuple(ind.movimentos) in elite_keys:
            continue
        nova.append(ind)

    while len(nova) < pop_size:
        nova.append(random.choice(candidatos))

    return nova


# ======================================================
# === Atualização da métrica delta ===
# ======================================================
def atualizar_delta(populacao, prev_stats):
    """
    Atualiza o valor de delta para todos os indivíduos da população.

    Processo:
    1. Calcula as médias populacionais de dmax e davg.
    2. Compara essas médias com a geração anterior.
    3. Escolhe a métrica que mais variou.
    4. Calcula delta individualmente usando a métrica ativa.

    Retorna um dicionário com as estatísticas da geração.
    """
    # Coleta das métricas individuais
    dmax_vals = np.array([ind.dmax for ind in populacao], dtype=float)
    davg_vals = np.array([ind.davg for ind in populacao], dtype=float)

    dmax_fin = dmax_vals[np.isfinite(dmax_vals)]
    davg_fin = davg_vals[np.isfinite(davg_vals)]

    # Médias populacionais da geração atual
    mean_dmax = float(np.mean(dmax_fin)) if dmax_fin.size else float("inf")
    mean_davg = float(np.mean(davg_fin)) if davg_fin.size else float("inf")

    # Valores da geração anterior
    active_metric = prev_stats.get("active_metric", "davg") if prev_stats else "davg"
    prev_dmax = float(prev_stats.get("mean_dmax", float("inf"))) if prev_stats else float("inf")
    prev_davg = float(prev_stats.get("mean_davg", float("inf"))) if prev_stats else float("inf")

    def razao(atual, anterior):
        """
        Calcula a razão entre o valor atual e o anterior.
        Retorna None quando a comparação não é válida.
        """
        if (not np.isfinite(atual)) or (not np.isfinite(anterior)) or anterior <= 0:
            return None
        return atual / anterior

    r_dmax = razao(mean_dmax, prev_dmax)
    r_davg = razao(mean_davg, prev_davg)

    # Escolhe a métrica que mais variou entre gerações
    active_metric = escolhe_metrica_variacao_abs(
        r_dmax,
        r_davg,
        atual=active_metric
    )

    # Cálculo de delta por indivíduo
    EPS = 1e-9
    for ind in populacao:
        d = ind.dmax if active_metric == "dmax" else ind.davg
        if d is None or (not np.isfinite(d)) or d <= 0:
            ind.delta = None
        else:
            ind.delta = ind.f * max(EPS, d)

    return {
        "mean_dmax": mean_dmax,
        "mean_davg": mean_davg,
        "active_metric": active_metric
    }


def escolhe_metrica_variacao_abs(r_dmax, r_davg, atual="davg"):
    """
    Escolhe entre dmax e davg com base na maior variação absoluta em relação a 1.

    Regras:
    - escolhe a métrica que mais variou entre gerações;
    - em caso de empate, mantém a métrica atual.
    """
    def variacao(r):
        if r is None:
            return -1.0
        return abs(r - 1.0)

    var_dmax = variacao(r_dmax)
    var_davg = variacao(r_davg)

    if var_dmax == var_davg:
        return atual

    return "dmax" if var_dmax > var_davg else "davg"


# ======================================================
# === Geração de gráfico da execução ===
# ======================================================
def salvar_grafico_melhor_fitness_com_marcador(df_hist, pasta_saida, nome_proteina, execucao_id):
    """
    Salva um gráfico da evolução do melhor fitness ao longo das gerações,
    destacando a geração em que ocorreu o melhor valor.
    """
    if df_hist is None or df_hist.empty:
        print("[WARN] df_hist vazio.")
        return None

    if "Iteracao" not in df_hist.columns or "Melhor Fitness" not in df_hist.columns:
        print("[WARN] df_hist sem colunas necessárias.")
        return None

    xs = df_hist["Iteracao"].to_numpy()
    ys = df_hist["Melhor Fitness"].to_numpy()

    idx_best = int(df_hist["Melhor Fitness"].idxmax())
    best_gen = int(df_hist.loc[idx_best, "Iteracao"])
    best_fit = float(df_hist.loc[idx_best, "Melhor Fitness"])

    plt.figure(figsize=(8, 4.5))
    plt.plot(xs, ys)
    plt.axvline(best_gen, linestyle="--", linewidth=1)
    plt.scatter([best_gen], [best_fit], s=60)
    plt.xlabel("Geração")
    plt.ylabel("Melhor fitness (nH)")
    plt.title(f"{nome_proteina} — exec {execucao_id} — melhor gen {best_gen} (fit={best_fit:.0f})")
    plt.grid(True, alpha=0.3)

    ultima_gen = int(df_hist["Iteracao"].max())
    nome_arquivo = f"{nome_proteina}_exec{execucao_id:02d}_ate{ultima_gen:04d}_bestgen{best_gen:04d}.png"
    caminho_fig = os.path.join(pasta_saida, nome_arquivo)

    plt.tight_layout()
    plt.savefig(caminho_fig, dpi=200)
    plt.close()
    return caminho_fig


# ======================================================
# === Comparação e regra de mutação condicional ===
# ======================================================
def eh_melhor(ind, melhor):
    """
    Verifica se um indivíduo é melhor que a referência.

    Nesta implementação, a comparação está baseada em f.
    """
    if melhor is None:
        return True
    return ind.f > melhor.f


def precisa_mutar(ind, melhor):
    """
    Retorna True se o indivíduo não for melhor que a referência.
    Essa regra é usada para aplicar mutação apenas em descendentes
    que não superam o melhor indivíduo atual.
    """
    if melhor is None:
        return False
    return ind.f <= melhor.f


# ======================================================
# === Algoritmo Evolutivo Monoobjetivo com delta ===
# ======================================================
def aemo_hp_3d_sem_mc(
    cadeia_raw,
    pop_init=8,
    taxa_mut=0.2,
    rho=1.0,
    torneio_k=2,
    taxa_cross=1.0,
    max_avaliacoes=None,
    seed=None,
    elitismo=1,
    target_nH=None
):
    """
    Executa o algoritmo evolutivo monoobjetivo com uso da métrica delta.

    Estrutura geral:
    1. Gera população inicial aleatória
    2. Avalia os indivíduos
    3. Atualiza os valores de delta
    4. Seleciona pais por torneio
    5. Gera filhos por crossover
    6. Aplica mutação condicional
    7. Atualiza delta dos candidatos
    8. Seleciona sobreviventes no esquema (mu + lambda)

    Critérios de parada:
    - atingir o número máximo de avaliações;
    - atingir um alvo de nH, quando informado.
    """
    global GLOBAL_AVALIACOES

    cadeia = ''.join([c for c in cadeia_raw.strip().upper() if c in ('H', 'P')])
    n_res = len(cadeia)
    if n_res < 2:
        raise ValueError("Cadeia muito curta")

    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    # ---------------- População inicial ----------------
    populacao = []
    for _ in range(pop_init):
        if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
            break
        ind = gerar_individuo_aleatorio(n_res)
        avaliar_base(ind, cadeia, rho=rho)
        populacao.append(ind)

    if not populacao:
        raise RuntimeError("Nenhum indivíduo pôde ser avaliado (max_avaliacoes muito baixo).")

    # Melhor indivíduo global no critério atual
    melhor_global = max(populacao, key=lambda ind: ind.f)
    geracao_melhor = 0
    aval_ate_target = None

    # Estatísticas da geração anterior para atualização do delta
    prev_stats = None
    prev_stats = atualizar_delta(populacao, prev_stats)

    # Histórico da execução
    melhores_f = []
    medias_f = []
    desvios_f = []

    # Diversidade e estatísticas de energia por geração
    genotipos_unicos = []
    energia_media = []
    energia_dp = []

    def coletar_stats_pop(pop):
        """
        Coleta estatísticas da população:
        - número de genótipos únicos
        - energia média
        - desvio padrão da energia
        """
        unicos = len({tuple(ind.movimentos) for ind in pop})
        E = np.array([ind.energia for ind in pop], dtype=float)
        return unicos, float(np.mean(E)), float(np.std(E))

    # Registro da geração 0
    f_vals0 = np.array([ind.f for ind in populacao], dtype=float)
    medias_f.append(float(np.mean(f_vals0)))
    desvios_f.append(float(np.std(f_vals0)))
    melhor0 = max(populacao, key=lambda ind: ind.f)
    melhores_f.append(melhor0.f)

    u0, em0, edp0 = coletar_stats_pop(populacao)
    genotipos_unicos.append(u0)
    energia_media.append(em0)
    energia_dp.append(edp0)

    geracao = 0
    while True:
        if max_avaliacoes is not None and GLOBAL_AVALIACOES >= max_avaliacoes:
            break

        if target_nH is not None and melhor_global.nC == 0 and melhor_global.nH >= target_nH:
            if aval_ate_target is None:
                aval_ate_target = GLOBAL_AVALIACOES
            break

        # ---------- Geração de filhos ----------
        filhos = []
        while len(filhos) < pop_init:
            pai = selecao_torneio_delta(populacao, k=torneio_k)
            mae = selecao_torneio_delta(populacao, k=torneio_k)

            if random.random() < taxa_cross:
                f1, f2 = crossover_multipontos_2filhos(pai, mae, n_res)
            else:
                f1, f2 = Individuo(pai.movimentos[:]), Individuo(mae.movimentos[:])

            for filho in (f1, f2):
                if len(filhos) >= pop_init:
                    break

                # Avalia o filho antes da mutação
                avaliar_base(filho, cadeia, rho=rho)

                # Aplica mutação apenas se ele não superar o melhor global
                if precisa_mutar(filho, melhor_global):
                    filho = mutacao_por_gene(filho, taxa_mut)
                    avaliar_base(filho, cadeia, rho=rho)

                filhos.append(filho)

        # ---------- Atualização de delta ----------
        candidatos = populacao + filhos
        prev_stats = atualizar_delta(candidatos, prev_stats)

        # ---------- Seleção de sobreviventes ----------
        populacao = selecao_sobreviventes_mu_plus_lambda_delta(
            populacao, filhos, pop_size=pop_init, elitismo=elitismo
        )

        # ---------- Estatísticas da geração ----------
        f_vals = np.array([ind.f for ind in populacao], dtype=float)
        medias_f.append(float(np.mean(f_vals)))
        desvios_f.append(float(np.std(f_vals)))

        melhor_ger = max(populacao, key=lambda ind: ind.f)
        melhores_f.append(melhor_ger.f)

        u, em, edp = coletar_stats_pop(populacao)
        genotipos_unicos.append(u)
        energia_media.append(em)
        energia_dp.append(edp)

        if melhor_ger.f > melhor_global.f:
            melhor_global = melhor_ger
            geracao_melhor = geracao + 1

        geracao += 1

    resumo = {
        "Melhor Fitness (nH penalizado se colisão)": melhor_global.f,
        "nH(c)": melhor_global.nH,
        "nC(c)": melhor_global.nC,
        "Geração do Melhor": geracao_melhor,
        "Aval_ate_target": aval_ate_target,
        "Aval_total": GLOBAL_AVALIACOES,
    }

    df_hist = pd.DataFrame({
        "Iteracao": list(range(len(melhores_f))),
        "Genotipos_unicos": genotipos_unicos,
        "Fitness Medio": medias_f,
        "Fitness DP": desvios_f,
        "Melhor Fitness": melhores_f,
        "Energia Media (pop)": energia_media,
        "Energia DP (pop)": energia_dp,
    })

    caminho_fig = salvar_grafico_melhor_fitness_com_marcador(
        df_hist,
        pasta_saida,
        nome_proteina,
        i + 1
    )
    print(f"[OK] Execução {i+1}: gráfico salvo em {caminho_fig}")

    return df_hist, resumo, melhor_global, populacao


# ======================================================
# === Execução repetida dos experimentos ===
# ======================================================
if __name__ == "__main__":
    # Arquivo da proteína e diretório de saída
    caminho_arquivo = "/content/drive/MyDrive/pastaOrigem/273d.10.txt"
    pasta_saida = "/content/drive/MyDrive/pastaDestino/273d10"
    os.makedirs(pasta_saida, exist_ok=True)

    # Leitura da instância:
    # linha 1 -> número de resíduos
    # linha 2 -> cadeia HP
    with open(caminho_arquivo) as f:
        n = int(f.readline().strip())
        cadeia = ''.join([c for c in f.readline().strip().upper() if c in ('H', 'P')])
        assert len(cadeia) == n

    nome_proteina = os.path.splitext(os.path.basename(caminho_arquivo))[0]

    # Ótimo conhecido da instância
    ENERGIA_OTIMA = -11

    if n <= 36:
        # Parâmetros experimentais para proteínas pequenas
        taxa_mut = 0.03
        taxa_cross = 0.80
        MAX_AVALIACOES = 1_500_000

    NUM_EXECUCOES = 50
    resultados_execucoes = []
    hists_execucoes = []

    for i in range(NUM_EXECUCOES):
        GLOBAL_AVALIACOES = 0
        random.seed()
        np.random.seed()

        print(f"\n========== EXECUÇÃO {i+1}/{NUM_EXECUCOES} ==========")

        # Como energia = -nH em soluções factíveis, o alvo em nH é o oposto da energia ótima
        NH_OTIMO = -ENERGIA_OTIMA

        df_hist, resumo, melhor, pop_final = aemo_hp_3d_sem_mc(
            cadeia,
            pop_init=120,
            taxa_mut=taxa_mut,
            rho=1.0,
            torneio_k=2,
            taxa_cross=taxa_cross,
            max_avaliacoes=MAX_AVALIACOES,
            elitismo=1,
            target_nH=NH_OTIMO,
        )

        # Estatísticas da população final desta execução
        gen_unicos_final = len({tuple(ind.movimentos) for ind in pop_final})
        E_final = np.array([ind.energia for ind in pop_final], dtype=float)
        E_media_final = float(np.mean(E_final))
        E_dp_final = float(np.std(E_final))

        # Energia equivalente só faz sentido para solução factível
        melhor_energia = (-melhor.nH) if melhor.nC == 0 else float("inf")

        hit_otimo = (
            (resumo["Aval_ate_target"] is not None) and
            (melhor.nC == 0) and
            (melhor.nH >= NH_OTIMO)
        )
        aval_ate_otimo = resumo["Aval_ate_target"]

        resultados_execucoes.append({
            "Execucao": i + 1,
            "Melhor_Energia": melhor_energia,
            "Hit_otimo": hit_otimo,
            "Aval_ate_otimo": aval_ate_otimo,
            "Aval_total": resumo["Aval_total"],
            "Melhor_fitness": resumo["Melhor Fitness (nH penalizado se colisão)"],
            "nH": melhor.nH,
            "nC": melhor.nC,
            "Genotipos_unicos_final": gen_unicos_final,
            "Energia_media_finalpop": E_media_final,
            "Energia_dp_finalpop": E_dp_final,
        })

        hists_execucoes.append(df_hist)

    df_resumo = pd.DataFrame(resultados_execucoes)

    # Melhor execução: prioriza energia finita; se não houver, usa maior fitness
    df_feasible = df_resumo.replace([np.inf, -np.inf], np.nan).dropna(subset=["Melhor_Energia"])
    if not df_feasible.empty:
        idx_best = int(df_feasible["Melhor_Energia"].idxmin())
    else:
        idx_best = int(df_resumo["Melhor_fitness"].idxmax())

    df_hist_best = hists_execucoes[idx_best]
    best_exec_num = int(df_resumo.loc[idx_best, "Execucao"])

    # ======================================================
    # === Estatísticas da melhor energia por execução ===
    # ======================================================
    energies = df_resumo["Melhor_Energia"].replace([np.inf, -np.inf], np.nan).dropna()

    melhor_energia_media = float(energies.mean()) if not energies.empty else float("nan")
    melhor_energia_dp = float(energies.std(ddof=1)) if len(energies) > 1 else float("nan")

    qtd_facteis = int(energies.shape[0])
    qtd_total = int(df_resumo.shape[0])

    print("\n\n===== RESUMO DAS 50 EXECUÇÕES =====")
    print(df_resumo)

    # Considera apenas execuções que atingiram o ótimo
    df_sucesso = df_resumo[df_resumo["Hit_otimo"] == True]

    if not df_sucesso.empty:
        media_aval = df_sucesso["Aval_ate_otimo"].mean()
        mediana_aval = df_sucesso["Aval_ate_otimo"].median()
    else:
        media_aval = float('nan')
        mediana_aval = float('nan')

    taxa_acertos = 100.0 * df_sucesso.shape[0] / NUM_EXECUCOES

    print("\n===== RESUMO GLOBAL (estilo artigo) =====")
    print(f"Ótimo conhecido: {ENERGIA_OTIMA}")
    print(f"Menor energia obtida: {df_resumo['Melhor_Energia'].min()}")
    print(f"Acurácia (Ac.): {taxa_acertos:.1f} %")
    print(f"Média Aval. (apenas execuções com ótimo): {media_aval:.0f}")
    print(f"Mediana Aval.: {mediana_aval:.0f}")

    # Salva o resumo em arquivo texto
    caminho_saida_txt = os.path.join(pasta_saida, f"{nome_proteina}.txt")
    with open(caminho_saida_txt, "w") as f_out:
        f_out.write(f"Arquivo da proteína: {caminho_arquivo}\n")
        f_out.write(f"Nome da proteína (base): {nome_proteina}\n")
        f_out.write(f"Ótimo conhecido: {ENERGIA_OTIMA}\n\n")

        f_out.write("===== RESUMO DAS 50 EXECUÇÕES =====\n")
        f_out.write(df_resumo.to_string(index=False))
        f_out.write("\n\n===== RESUMO GLOBAL (estilo artigo) =====\n")
        f_out.write(f"Menor energia obtida: {df_resumo['Melhor_Energia'].min()}\n")
        f_out.write(f"Acurácia (Ac.): {taxa_acertos:.1f} %\n")
        f_out.write(f"Média Aval. (execuções com ótimo): {media_aval:.0f}\n")
        f_out.write(f"Mediana Aval.: {mediana_aval:.0f}\n")

        f_out.write("\n----- ESTATÍSTICAS (MELHOR ENERGIA POR EXECUÇÃO) -----\n")
        f_out.write(f"Melhor_Energia (média ± dp): {melhor_energia_media:.4f} ± {melhor_energia_dp:.4f}\n")
        f_out.write(f"Execuções factíveis consideradas: {qtd_facteis}/{qtd_total}\n")

    print(f"\nResumo salvo em: {caminho_saida_txt}")